# COMP4680/8650 Flow Matching on Colab

Before running: in Colab, choose `Runtime -> Change runtime type -> A100 GPU` or another GPU runtime.

This notebook is intentionally restart-safe: if `/content` is wiped, run cells from the top again. Outputs that matter are copied to Google Drive.

## 1. Mount Google Drive

Outputs are copied to Drive so they survive after the Colab runtime shuts down. This cell also handles the case where Drive is already mounted.

In [ ]:
from google.colab import drive
import os

if os.path.isdir('/content/drive/MyDrive'):
    print('Google Drive is already mounted.')
else:
    drive.mount('/content/drive', force_remount=True)

## 2. Clone or Update the Repository

Colab sees only files that have been pushed to GitHub. If you changed files locally, push them first, then run this cell.

In [ ]:
%cd /content

if not os.path.isdir('/content/COMP4680-8650-Flowmatching/.git'):
    !rm -rf /content/COMP4680-8650-Flowmatching
    !git clone https://github.com/Katherine1616/COMP4680-8650-Flowmatching.git /content/COMP4680-8650-Flowmatching
else:
    print('Repository already exists; pulling latest main.')

%cd /content/COMP4680-8650-Flowmatching
!git pull origin main
!git log -1 --oneline

## 3. Install Dependencies

In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']
!uv sync

## 4. Check Data and GPU

The toy `.npz` data files are now tracked in the repository. This cell verifies that the data and GPU are visible.

In [ ]:
%cd /content/COMP4680-8650-Flowmatching
!ls -lh data
!nvidia-smi

import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))

## 5. Optional: Run or Restore Part 2 Baseline Checkpoints

Part 4.2 MeanFlow can generate the required 9 MeanFlow figures without Part 2 checkpoints. Part 2 checkpoints are only needed for the extra standard-flow-matching comparison grids.

This cell first tries to restore Part 2 outputs from Drive. If they are not on Drive, it trains only the three needed `D=32, x-pred/v-loss` checkpoints.

In [ ]:
%cd /content/COMP4680-8650-Flowmatching
!mkdir -p outputs/part2

if os.path.isdir('/content/drive/MyDrive/COMP4680-8650-Flowmatching/outputs/part2'):
    !rsync -av /content/drive/MyDrive/COMP4680-8650-Flowmatching/outputs/part2/ outputs/part2/
else:
    print('No Part 2 outputs found on Drive; training the three D=32 x-pred/v-loss checkpoints now.')
    !uv run python scripts/part2.py --datasets swiss_roll,gaussians,circles --dims 32 --predictions x --losses v --skip-existing --no-progress

!find outputs/part2 -maxdepth 1 -name 'model_*_d32_x_pred_v_loss.pt' -print | sort

## 6. Run Part 4.2 MeanFlow

This trains MeanFlow at `D=32` for all three datasets and saves the required 9 figures: 3 datasets × 3 step counts (`1`, `2`, `5`). It also saves per-dataset grids and comparison grids when Part 2 checkpoints are available.

In [ ]:
%cd /content/COMP4680-8650-Flowmatching
!uv run python scripts/part4_meanflow.py --skip-existing --no-progress
!ls -lh outputs/part4/meanflow

## 7. Copy MeanFlow Outputs to Google Drive

Run this immediately after Part 4.2 finishes, before the Colab runtime disconnects.

In [ ]:
%cd /content/COMP4680-8650-Flowmatching
!mkdir -p /content/drive/MyDrive/COMP4680-8650-Flowmatching/outputs/part4/meanflow
!rsync -av outputs/part4/meanflow/ /content/drive/MyDrive/COMP4680-8650-Flowmatching/outputs/part4/meanflow/
!find /content/drive/MyDrive/COMP4680-8650-Flowmatching/outputs/part4/meanflow -maxdepth 1 -type f | sort

## 8. Optional: Run Full Part 2 Grid

Only run this if you need to regenerate all Part 2 experiments from scratch.

In [ ]:
%cd /content/COMP4680-8650-Flowmatching
!uv run python scripts/part2.py --skip-existing --no-progress
!mkdir -p /content/drive/MyDrive/COMP4680-8650-Flowmatching/outputs
!rsync -av outputs/part2/ /content/drive/MyDrive/COMP4680-8650-Flowmatching/outputs/part2/

## Useful Commands

Pull latest code after pushing from your laptop:

```bash
%cd /content/COMP4680-8650-Flowmatching
!git pull origin main
```

Quick Part 2 smoke test:

```bash
!uv run python scripts/part2.py --datasets swiss_roll --dims 2 --predictions x --losses x --train-steps 100 --num-samples 512 --max-experiments 1
```